# External spectrum-analyser envelope analysis

This notebook analyses CSV traces exported after `external_q_replay()` in
`resonator_coil_characterisation.ipynb`. It performs no hardware access.

The analyser records discrete impulses because its sweep and the Red Pitaya stepped tone are
asynchronous. The resonance is therefore fitted to the **impulse maxima**, after converting dBm
to linear power, using a Lorentzian envelope plus a constant background.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.signal import find_peaks


In [ ]:
def load_siglent_csv(path):
    """Return metadata, frequency in Hz, and level in dBm from a Siglent CSV."""
    path = Path(path)
    lines = path.read_text(encoding='utf-8-sig', errors='replace').splitlines()
    try:
        start = lines.index('Trace Data') + 1
    except ValueError:
        raise ValueError('CSV does not contain a Trace Data section')
    metadata = {}
    for line in lines[:start-1]:
        fields = [x.strip() for x in line.split(',')]
        if len(fields) >= 2 and fields[0]:
            metadata[fields[0]] = fields[1]
    rows = []
    for line in lines[start:]:
        fields = line.split(',')
        if len(fields) < 2:
            break
        try:
            rows.append((float(fields[0]), float(fields[1])))
        except ValueError:
            break
    if len(rows) < 20:
        raise ValueError('Trace Data contains fewer than 20 numeric points')
    data = np.asarray(rows, dtype=float)
    order = np.argsort(data[:, 0])
    f, dbm = data[order, 0], data[order, 1]
    if np.any(np.diff(f) <= 0):
        raise ValueError('trace frequencies must be unique and increasing')
    return metadata, f, dbm


def _grid_spacing(f_peaks, analyser_bin_hz=None):
    gaps = np.diff(f_peaks)
    if len(gaps) < 4:
        raise ValueError('too few peak spacings to identify the stepped-frequency grid')
    median = float(np.median(gaps))
    analyser_bin_hz = (float(analyser_bin_hz) if analyser_bin_hz is not None
                       else 0.0)
    # A tone step need not be an integer number of analyser bins. Selected
    # maxima can therefore alternate between adjacent rounded gaps (for
    # example 5.33 and 8.00 kHz for a roughly 6.94 kHz tone step).
    base_group = gaps[gaps < 1.5 * median]
    if analyser_bin_hz > 0 and len(base_group) >= 3:
        initial = float(np.mean(base_group))
        multiples = np.maximum(1, np.rint(gaps / initial))
        spacing = float(np.sum(gaps) / np.sum(multiples))
        consistent = np.abs(gaps - multiples*spacing) <= 0.70*analyser_bin_hz
    else:
        spacing = float(np.median(base_group if len(base_group) >= 3 else gaps))
        multiples = np.maximum(1, np.rint(gaps / spacing))
        residual = np.abs(gaps / multiples - spacing) / spacing
        consistent = residual <= 0.20
    if np.count_nonzero(consistent) < 0.80 * len(gaps):
        raise ValueError('impulse spacing is inconsistent; check max hold and replay timing')
    return spacing


def lorentzian_power(f, f0, linewidth, peak_dbm, floor_dbm):
    floor = 10.0 ** (floor_dbm / 10.0)
    peak = 10.0 ** (peak_dbm / 10.0)
    return floor + peak / (1.0 + (2.0 * (f - f0) / linewidth) ** 2)


def fit_impulse_envelope(path, prominence_db=8.0, min_distance_bins=2,
                         min_peaks=10):
    """Extract impulse maxima and fit a Lorentzian power envelope.

    Returns a dictionary containing the raw trace, selected maxima, fitted
    parameters, diagnostics, and analyser metadata.
    """
    metadata, f, dbm = load_siglent_csv(path)
    peaks, _ = find_peaks(dbm, prominence=float(prominence_db),
                          distance=int(min_distance_bins))
    if dbm[0] > dbm[1] + prominence_db:
        peaks = np.r_[0, peaks]
    if dbm[-1] > dbm[-2] + prominence_db:
        peaks = np.r_[peaks, len(dbm)-1]
    peaks = np.unique(peaks)
    if len(peaks) < int(min_peaks):
        raise ValueError('fewer than %d clear impulses; repeat max hold for longer' % min_peaks)
    fp, yp = f[peaks], dbm[peaks]
    spacing = _grid_spacing(fp, np.median(np.diff(f)))

    def model_db(x):
        return 10.0*np.log10(lorentzian_power(fp, x[0], x[1], x[2], x[3]))

    floor0 = float(np.percentile(dbm, 20))
    x0 = np.array([fp[np.argmax(yp)], max(np.ptp(fp)/4.0, 2*spacing),
                   float(yp.max()), floor0])
    lower = [f.min(), spacing/2.0, -140.0, -160.0]
    upper = [f.max(), 2.0*np.ptp(f), 40.0, float(yp.max()-1e-3)]
    result = least_squares(lambda x: model_db(x)-yp, x0, bounds=(lower, upper),
                           loss='soft_l1', f_scale=1.0)
    f0, linewidth, peak_dbm, floor_dbm = map(float, result.x)
    if not result.success or not np.isfinite(result.x).all():
        raise RuntimeError('Lorentzian envelope fit did not converge')
    if not (f.min() < f0 < f.max()):
        raise ValueError('fitted resonance lies outside the analyser sweep')
    if linewidth < 2.0*spacing:
        raise ValueError('linewidth is unresolved by the stepped-frequency grid')
    if min(f0-fp.min(), fp.max()-f0) < linewidth:
        raise ValueError('impulse envelope does not cover at least one linewidth on both sides')
    fitted_peaks = model_db(result.x)
    rms_db = float(np.sqrt(np.mean((fitted_peaks-yp)**2)))
    dense_f = np.linspace(f.min(), f.max(), 1200)
    dense_dbm = 10.0*np.log10(lorentzian_power(
        dense_f, result.x[0], result.x[1], result.x[2], result.x[3]))
    return {'path': str(path), 'metadata': metadata, 'f': f, 'dbm': dbm,
            'peak_indices': peaks, 'peak_f': fp, 'peak_dbm': yp,
            'fit_f': dense_f, 'fit_dbm': dense_dbm, 'f0': f0,
            'linewidth': linewidth, 'Q': f0/linewidth, 'spacing': spacing,
            'n_peaks': int(len(peaks)), 'rms_db': rms_db,
            'peak_level_dbm': peak_dbm, 'floor_dbm': floor_dbm}


def print_envelope_result(result):
    print(Path(result['path']).name)
    print('  f_0       = %.6f MHz' % (result['f0']/1e6))
    print('  FWHM      = %.3f kHz' % (result['linewidth']/1e3))
    print('  Q         = %.1f' % result['Q'])
    print('  impulses  = %d, spacing %.3f kHz' %
          (result['n_peaks'], result['spacing']/1e3))
    print('  fit RMS   = %.3f dB' % result['rms_db'])


def fit_single_shift(path, reference_f0=39.939722e6,
                     reference_linewidth=222.045e3):
    """Fit one trace and report its shift from the open-loop resonance."""
    result = fit_impulse_envelope(path)
    reference_f0 = float(reference_f0)
    reference_linewidth = float(reference_linewidth)
    if not np.isfinite(reference_f0) or not np.isfinite(reference_linewidth):
        raise ValueError('reference frequency and linewidth must be finite')
    if reference_linewidth <= 0:
        raise ValueError('reference linewidth must be positive')
    delta_f = result['f0']-reference_f0
    at_peak = 10.0*np.log10(lorentzian_power(
        result['f0'], result['f0'], result['linewidth'],
        result['peak_level_dbm'], result['floor_dbm']))
    at_reference = 10.0*np.log10(lorentzian_power(
        reference_f0, result['f0'], result['linewidth'],
        result['peak_level_dbm'], result['floor_dbm']))
    result.update({'reference_f0': reference_f0,
                   'reference_linewidth': reference_linewidth,
                   'delta_f': delta_f,
                   'shift_original_linewidths': delta_f/reference_linewidth,
                   'drop_at_original_db': at_peak-at_reference})
    print_envelope_result(result)
    print('  delta f   = %+.3f kHz' % (delta_f/1e3))
    print('  shift/LW0 = %+.3f' % result['shift_original_linewidths'])
    print('  drop at original f_0 = %.3f dB' % result['drop_at_original_db'])
    return result


def plot_envelope(result, ax=None):
    """Plot complete CSV, accepted impulses, and fitted envelope."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9.2, 5.2))
    ax.plot(result['f']/1e6, result['dbm'], lw=0.8, alpha=0.75,
            label='CSV max-hold trace')
    ax.scatter(result['peak_f']/1e6, result['peak_dbm'], s=20, zorder=3,
               label='accepted impulse maxima')
    ax.plot(result['fit_f']/1e6, result['fit_dbm'], lw=2.2,
            label='Lorentzian envelope')
    ax.axvline(result['f0']/1e6, lw=1.0, ls=':', color='0.35')
    ax.set_title('Q = %.1f,  f_0 = %.6f MHz' % (result['Q'], result['f0']/1e6))
    ax.set_xlabel('frequency / MHz'); ax.set_ylabel('level / dBm')
    ax.grid(True, alpha=0.25); ax.legend()
    return ax


def plot_two_envelopes_measured(first, second, first_label, second_label, ax=None):
    """Overlay two fitted Lorentzian envelopes on their measured-dBm scale."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9.4, 5.4))
    colours = ('#0072B2', '#D55E00')
    for result, label, colour in zip((first, second),
                                      (first_label, second_label), colours):
        ax.plot(result['fit_f']/1e6, result['fit_dbm'], lw=2.4, color=colour,
                label='%s: f$_0$ = %.6f MHz, Q = %.1f' %
                      (label, result['f0']/1e6, result['Q']))
        ax.axvline(result['f0']/1e6, color=colour, lw=1.2, ls=':')
    delta_khz = (first['f0']-second['f0'])/1e3
    margin = 2.5*max(first['linewidth'], second['linewidth'])
    ax.set_xlim((min(first['f0'], second['f0'])-margin)/1e6,
                (max(first['f0'], second['f0'])+margin)/1e6)
    ax.text(0.02, 0.04, r'$\Delta f_0$ = %+.3f kHz' % delta_khz,
            transform=ax.transAxes, ha='left', va='bottom',
            bbox={'boxstyle': 'round,pad=0.35', 'facecolor': 'white',
                  'edgecolor': '0.65', 'alpha': 0.92})
    ax.set_xlabel('frequency / MHz'); ax.set_ylabel('level / dBm')
    ax.set_title('Gain 0.9: phase-shifted and Q-enhanced Lorentzian fits')
    ax.grid(True, alpha=0.25); ax.legend(loc='best')
    return ax


def plot_same_frequency_comparison(q_fit, shifted_fit, q_coefficients,
                                   shifted_coefficients, ax=None):
    """Compare measured-dBm fits at the Q-enhanced resonant frequency."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10.4, 6.2))
    q_ci, q_cq = map(float, q_coefficients)
    s_ci, s_cq = map(float, shifted_coefficients)
    q_mag, s_mag = np.hypot(q_ci, q_cq), np.hypot(s_ci, s_cq)
    q_angle = np.degrees(np.arctan2(q_cq, q_ci))
    s_angle = np.degrees(np.arctan2(s_cq, s_ci))
    compare_f = q_fit['f0']
    def fitted_level(result, frequency):
        return float(10.0*np.log10(lorentzian_power(
            frequency, result['f0'], result['linewidth'],
            result['peak_level_dbm'], result['floor_dbm'])))
    q_level = fitted_level(q_fit, compare_f)
    shifted_level = fitted_level(shifted_fit, compare_f)
    difference_db = q_level-shifted_level
    colours = {'q': '#D55E00', 'shift': '#0072B2'}
    overlap_lo = max(q_fit['f'].min(), shifted_fit['f'].min())
    overlap_hi = min(q_fit['f'].max(), shifted_fit['f'].max())
    for result, colour, marker_label in (
            (q_fit, colours['q'], 'Q-enhanced impulse maxima'),
            (shifted_fit, colours['shift'], 'phase 52.1 deg impulse maxima')):
        keep = ((result['peak_f'] >= overlap_lo) &
                (result['peak_f'] <= overlap_hi))
        ax.scatter(result['peak_f'][keep]/1e6, result['peak_dbm'][keep],
                   s=19, alpha=0.58, color=colour, label=marker_label)
        dense_keep = ((result['fit_f'] >= overlap_lo) &
                      (result['fit_f'] <= overlap_hi))
        ax.plot(result['fit_f'][dense_keep]/1e6, result['fit_dbm'][dense_keep],
                lw=2.5, color=colour)
    x_mhz = compare_f/1e6
    ax.axvline(x_mhz, color='0.25', lw=1.1, ls=':',
               label='Q-enhanced $f_0$ = %.6f MHz' % x_mhz)
    ax.scatter([x_mhz, x_mhz], [q_level, shifted_level], s=58,
               color=[colours['q'], colours['shift']], edgecolor='white', zorder=6)
    ax.plot([x_mhz, x_mhz], [shifted_level, q_level], color='0.15', lw=2.0, zorder=5)
    ax.annotate('%.3f dB at %.6f MHz' % (difference_db, x_mhz),
                xy=(x_mhz, 0.5*(q_level+shifted_level)), xytext=(12, 0),
                textcoords='offset points', va='center', ha='left',
                bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white',
                      'edgecolor': '0.55', 'alpha': 0.94})
    q_label = ('Q-enhanced: |C| %.3f, angle %.3f deg, f0 %.6f MHz, '
               'FWHM %.3f kHz, Q %.1f, RMS %.3f dB' %
               (q_mag, q_angle, q_fit['f0']/1e6, q_fit['linewidth']/1e3,
                q_fit['Q'], q_fit['rms_db']))
    s_label = ('Phase 52.1 deg: |C| %.3f, angle %.3f deg, f0 %.6f MHz, '
               'FWHM %.3f kHz, Q %.1f, RMS %.3f dB' %
               (s_mag, s_angle, shifted_fit['f0']/1e6,
                shifted_fit['linewidth']/1e3, shifted_fit['Q'],
                shifted_fit['rms_db']))
    ax.text(0.015, 0.02, q_label+'\n'+s_label, transform=ax.transAxes,
            ha='left', va='bottom', fontsize=8.5,
            bbox={'boxstyle': 'round,pad=0.35', 'facecolor': 'white',
                  'edgecolor': '0.7', 'alpha': 0.92})
    ax.set_xlim(overlap_lo/1e6, overlap_hi/1e6)
    ax.set_xlabel('frequency / MHz'); ax.set_ylabel('level / dBm')
    ax.set_title('Gain 0.9: same-frequency Q-enhanced and phase-shifted comparison')
    ax.grid(True, alpha=0.25); ax.legend(loc='upper left', fontsize=8.5)
    return {'frequency': compare_f, 'q_level_dbm': q_level,
            'shifted_level_dbm': shifted_level,
            'difference_db': difference_db,
            'q_coefficient_magnitude': q_mag,
            'shifted_coefficient_magnitude': s_mag,
            'coefficient_magnitude_difference_db': 20*np.log10(q_mag/s_mag),
            'coefficient_angle_difference_deg': s_angle-q_angle}


def calibrate_to_open_loop(result, reference):
    """Return resonance and level metrics relative to one open-loop fit."""
    def level_at(item, frequency):
        return float(10.0*np.log10(lorentzian_power(
            frequency, item['f0'], item['linewidth'],
            item['peak_level_dbm'], item['floor_dbm'])))
    delta_f = result['f0']-reference['f0']
    return {'delta_f': delta_f,
            'shift_open_linewidths': delta_f/reference['linewidth'],
            'q_ratio': result['Q']/reference['Q'],
            'peak_change_db': (level_at(result, result['f0'])-
                               level_at(reference, reference['f0'])),
            'level_at_open_f0_dbm': level_at(result, reference['f0'])}


def plot_calibrated_three_way(reference, q_fit, shifted_fit, ax=None):
    """Plot open-loop, Q-enhanced and shifted fits on one measured-dBm axis."""
    if ax is None:
        _, ax = plt.subplots(figsize=(11.0, 6.5))
    items = ((reference, 'Open loop, gain 0', '#333333'),
             (q_fit, 'Q-enhanced, gain 0.9', '#D55E00'),
             (shifted_fit, 'Phase 52.1 deg, gain 0.9', '#0072B2'))
    overlap_lo = max(item[0]['f'].min() for item in items)
    overlap_hi = min(item[0]['f'].max() for item in items)
    for result, label, colour in items:
        peaks = ((result['peak_f'] >= overlap_lo) &
                 (result['peak_f'] <= overlap_hi))
        dense = ((result['fit_f'] >= overlap_lo) &
                 (result['fit_f'] <= overlap_hi))
        calibrated = calibrate_to_open_loop(result, reference)
        legend = ('%s: f0 %.6f MHz, Q %.1f, delta f %+.3f kHz' %
                  (label, result['f0']/1e6, result['Q'],
                   calibrated['delta_f']/1e3))
        ax.scatter(result['peak_f'][peaks]/1e6, result['peak_dbm'][peaks],
                   s=18, alpha=0.50, color=colour)
        ax.plot(result['fit_f'][dense]/1e6, result['fit_dbm'][dense],
                lw=2.5, color=colour, label=legend)
        ax.axvline(result['f0']/1e6, color=colour, lw=0.9, ls=':', alpha=0.7)
    compare_f = q_fit['f0']
    def level_at_comparison_frequency(item):
        return float(10.0*np.log10(lorentzian_power(
            compare_f, item['f0'], item['linewidth'],
            item['peak_level_dbm'], item['floor_dbm'])))
    levels = {name: level_at_comparison_frequency(item) for name, item in
              (('open', reference), ('q', q_fit), ('shifted', shifted_fit))}
    x_mhz = compare_f/1e6
    ax.scatter([x_mhz]*3, [levels['open'], levels['q'], levels['shifted']],
               s=54, color=['#333333', '#D55E00', '#0072B2'],
               edgecolor='white', zorder=7)
    ax.plot([x_mhz, x_mhz], [levels['shifted'], levels['q']],
            color='0.15', lw=2.0, zorder=6)
    difference_db = levels['q']-levels['shifted']
    ax.annotate('Q-enhanced - shifted = %.3f dB\nat %.6f MHz' %
                (difference_db, x_mhz),
                xy=(x_mhz, 0.5*(levels['q']+levels['shifted'])),
                xytext=(12, 0), textcoords='offset points', va='center',
                bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white',
                      'edgecolor': '0.55', 'alpha': 0.94})
    ax.set_xlim(overlap_lo/1e6, overlap_hi/1e6)
    ax.set_xlabel('frequency / MHz'); ax.set_ylabel('level / dBm')
    ax.set_title('Open-loop calibrated resonator comparison')
    ax.grid(True, alpha=0.25); ax.legend(loc='upper left', fontsize=8.5)
    return {'reference': calibrate_to_open_loop(reference, reference),
            'q_enhanced': calibrate_to_open_loop(q_fit, reference),
            'shifted': calibrate_to_open_loop(shifted_fit, reference),
            'levels_at_q_f0_dbm': levels,
            'q_minus_shifted_at_q_f0_db': difference_db}


In [ ]:
def fit_shift_campaign(entries):
    """Fit phase/gain traces and measure f0 relative to one reference trace."""
    if not isinstance(entries, (list, tuple)) or not entries:
        raise ValueError('SHIFT_TRACES must be a non-empty list')
    references = [item for item in entries if bool(item.get('reference', False))]
    if len(references) != 1:
        raise ValueError('exactly one SHIFT_TRACES entry must have reference=True')
    seen, results = set(), []
    for number, item in enumerate(entries, 1):
        for key in ('phase_deg', 'gain_frac', 'path'):
            if key not in item:
                raise ValueError('campaign entry %d is missing %s' % (number, key))
        phase = float(item['phase_deg']) % 360.0
        gain = float(item['gain_frac'])
        if not np.isfinite(phase) or not np.isfinite(gain) or not (0 <= gain <= 0.95):
            raise ValueError('campaign phase/gain is invalid at entry %d' % number)
        key = (round(phase, 9), round(gain, 9))
        if key in seen:
            raise ValueError('duplicate phase/gain campaign setting: %r' % (key,))
        if bool(item.get('clipped', False)):
            raise ValueError('entry %d is marked clipped; discard and reacquire it' % number)
        seen.add(key)
        fit = fit_impulse_envelope(item['path'])
        results.append({'phase_deg': phase, 'gain_frac': gain,
                        'reference': bool(item.get('reference', False)),
                        'path': str(item['path']), 'fit': fit,
                        'f0': fit['f0'], 'linewidth': fit['linewidth'],
                        'Q': fit['Q'], 'rms_db': fit['rms_db']})
    reference = next(row for row in results if row['reference'])
    reference_peak_power = lorentzian_power(
        reference['f0'], reference['f0'], reference['linewidth'],
        reference['fit']['peak_level_dbm'], reference['fit']['floor_dbm'])
    for row in results:
        row['delta_f'] = row['f0']-reference['f0']
        row['shift_original_linewidths'] = row['delta_f']/reference['linewidth']
        own_peak_power = lorentzian_power(
            row['f0'], row['f0'], row['linewidth'],
            row['fit']['peak_level_dbm'], row['fit']['floor_dbm'])
        power_at_original = lorentzian_power(
            reference['f0'], row['f0'], row['linewidth'],
            row['fit']['peak_level_dbm'], row['fit']['floor_dbm'])
        row['drop_at_original_db'] = max(
            0.0, 10.0*np.log10(own_peak_power/power_at_original))
        row['absolute_change_at_original_db'] = 10.0*np.log10(
            power_at_original/reference_peak_power)
    return results


def select_maximum_shifts(results, max_phase_gap_deg=90.0):
    """Select each gain's phase by maximum drop at the original resonance.

    The selected row also carries the signed centre shift in units of the
    zero-feedback reference linewidth. A near-complete phase span is required;
    otherwise a sampled edge could be mistaken for the true maximum.
    """
    groups = {}
    for row in results:
        if row['reference'] or row['gain_frac'] == 0:
            continue
        groups.setdefault(row['gain_frac'], []).append(row)
    if not groups:
        raise ValueError('campaign contains no non-zero gain phase scans')
    selected = []
    for gain in sorted(groups):
        rows = groups[gain]
        phases = sorted(row['phase_deg'] % 360.0 for row in rows)
        if len(phases) < 6:
            raise ValueError('gain %.3f has fewer than six distinct phases' % gain)
        circular_gaps = np.diff(phases+[phases[0]+360.0])
        if np.max(circular_gaps) > float(max_phase_gap_deg):
            raise ValueError('gain %.3f does not cover a complete phase span' % gain)
        best = max(rows, key=lambda row: row['drop_at_original_db'])
        selected.append(best)
    return selected


def print_maximum_shifts(results):
    selected = select_maximum_shifts(results)
    print('%8s %9s %14s %13s %15s' %
          ('gain', 'phase', 'shift / kHz', 'shift / LW0', 'drop at f0 / dB'))
    for row in selected:
        print('%8.3f %8.1f? %+14.3f %+13.4f %15.3f' %
              (row['gain_frac'], row['phase_deg'], row['delta_f']/1e3,
               row['shift_original_linewidths'], row['drop_at_original_db']))
    return selected


def plot_phase_shift_scan(results):
    """Show both shift/LW0 and original-frequency drop versus phase."""
    selected = select_maximum_shifts(results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), constrained_layout=True)
    gains = sorted(set(row['gain_frac'] for row in results
                       if not row['reference'] and row['gain_frac'] > 0))
    for gain in gains:
        rows = sorted((row for row in results if row['gain_frac'] == gain and
                       not row['reference']), key=lambda row: row['phase_deg'])
        phase = [row['phase_deg'] for row in rows]
        axes[0].plot(phase, [row['shift_original_linewidths'] for row in rows],
                     'o-', label='gain %.3f' % gain)
        axes[1].plot(phase, [row['drop_at_original_db'] for row in rows],
                     'o-', label='gain %.3f' % gain)
    for row in selected:
        axes[1].scatter([row['phase_deg']], [row['drop_at_original_db']],
                        marker='*', s=130, zorder=5)
    axes[0].axhline(0, color='0.4', lw=1)
    axes[0].set_ylabel('centre shift / original linewidth')
    axes[1].set_ylabel('fitted drop at original f_0 / dB')
    for ax in axes:
        ax.set_xlabel('feedback phase / degrees'); ax.grid(True, alpha=0.25)
        ax.legend()
    return fig, axes


def recommend_phase_refinement(results, half_width_deg=30.0, step_deg=5.0):
    """Return finer phase settings around each coarse maximum."""
    half_width_deg, step_deg = float(half_width_deg), float(step_deg)
    if half_width_deg <= 0 or step_deg <= 0 or step_deg > half_width_deg:
        raise ValueError('refinement width/step are invalid')
    settings = []
    for row in select_maximum_shifts(results):
        offsets = np.arange(-half_width_deg, half_width_deg+0.5*step_deg, step_deg)
        for offset in offsets:
            settings.append({'phase_deg': float((row['phase_deg']+offset) % 360.0),
                             'gain_frac': row['gain_frac']})
    return settings


def print_shift_campaign(results):
    print('%9s %8s %14s %12s %9s %10s' %
          ('phase', 'gain', 'f_0 / MHz', 'shift / kHz', 'Q', 'drop / dB'))
    for row in results:
        print('%8.1f? %8.3f %14.6f %+12.3f %9.1f %10.3f%s' %
              (row['phase_deg'], row['gain_frac'], row['f0']/1e6,
               row['delta_f']/1e3, row['Q'], row['drop_at_original_db'],
               '  REF' if row['reference'] else ''))


def plot_shift_envelopes(results):
    """Plot the fitted analyser trace for every campaign setting."""
    count = len(results); columns = min(2, count); rows = int(np.ceil(count/columns))
    fig, axes = plt.subplots(rows, columns, figsize=(9.2*columns, 4.8*rows),
                             squeeze=False, constrained_layout=True)
    for ax, row in zip(axes.ravel(), results):
        plot_envelope(row['fit'], ax=ax)
        ax.set_title('phase %.1f?, gain %.3f: f_0 %.6f MHz, Q %.1f' %
                     (row['phase_deg'], row['gain_frac'], row['f0']/1e6, row['Q']))
    for ax in axes.ravel()[count:]:
        ax.set_visible(False)
    return fig, axes


def plot_shift_campaign(results):
    """Plot absolute resonance frequency and reference-relative shift."""
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5), constrained_layout=True)
    phases = sorted(set(row['phase_deg'] for row in results))
    for phase in phases:
        rows = sorted((row for row in results if row['phase_deg'] == phase),
                      key=lambda row: row['gain_frac'])
        gain = [row['gain_frac'] for row in rows]
        axes[0].plot(gain, [row['f0']/1e6 for row in rows], 'o-', label='%.1f?' % phase)
        axes[1].plot(gain, [row['delta_f']/1e3 for row in rows], 'o-', label='%.1f?' % phase)
    axes[0].set_xlabel('gain / safe ceiling'); axes[0].set_ylabel('f_0 / MHz')
    axes[1].set_xlabel('gain / safe ceiling'); axes[1].set_ylabel('frequency shift / kHz')
    for ax in axes:
        ax.grid(True, alpha=0.25); ax.legend(title='phase')
    return fig, axes


def export_shift_results(results, path):
    """Write the compact numeric campaign table as CSV."""
    import csv
    fields = ('phase_deg', 'gain_frac', 'reference', 'f0_hz', 'delta_f_hz',
              'linewidth_hz', 'Q', 'rms_db', 'path')
    with open(path, 'w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields); writer.writeheader()
        for row in results:
            writer.writerow({'phase_deg': row['phase_deg'], 'gain_frac': row['gain_frac'],
                'reference': row['reference'], 'f0_hz': row['f0'],
                'delta_f_hz': row['delta_f'], 'linewidth_hz': row['linewidth'],
                'Q': row['Q'], 'rms_db': row['rms_db'], 'path': row['path']})
    return str(path)


## Analyse exported traces

Set paths to the CSV files exported by the analyser. Raw data stays outside the repository.
The result plot deliberately shows the impulses as well as the fitted envelope.


In [ ]:
TRACE_PATHS = [
    # r"D:\f 0.0 v2.csv",
    # r"D:\f 0.9 v2.csv",
]

results = []
for path in TRACE_PATHS:
    result = fit_impulse_envelope(path)
    results.append(result)
    print_envelope_result(result)
    plot_envelope(result)
    plt.show()


## Measure resonance shifts across phase and gain

List the exported traces with the phase and safe-ceiling gain fraction used during acquisition.
Exactly one entry must be marked as the reference. The campaign reports every fitted centre and
its shift relative to that reference.


In [ ]:
SHIFT_TRACES = [
    # {'phase_deg': 0.0, 'gain_frac': 0.0, 'path': r'path/to/reference.csv',
    #  'reference': True},
    # {'phase_deg': 90.0, 'gain_frac': 0.6, 'path': r'path/to/shift_p090p0_g0p600.csv'},
]

shift_results = []
if SHIFT_TRACES:
    shift_results = fit_shift_campaign(SHIFT_TRACES)
    print_shift_campaign(shift_results)
    print_maximum_shifts(shift_results)
    plot_phase_shift_scan(shift_results); plt.show()
    plot_shift_envelopes(shift_results); plt.show()
    plot_shift_campaign(shift_results); plt.show()
    # export_shift_results(shift_results, 'external_shift_results.csv')
